# Date Table for Data Warehouse

This notebook creates a comprehensive **date dimension table** for the data warehouse.

## Purpose
A date table (date dimension) is a central reference table in a data warehouse, used for time intelligence calculations and time-based analyses in Power BI and other BI tools.

## What does this notebook do?

1. **Defines a date range** (1900-2222)
2. **Creates the schema `dim`** in the data warehouse
3. **Generates a base table** with all date values in the defined range
4. **Extends the table** with calculated columns for:
   - Calendar hierarchies (year, month, quarter, week)
   - Time-based offsets (difference from today)
   - Weekdays and workdays
   - Helper fields for sorting and formatting

## Target Location
- **Catalog**: `thekitchen`
- **Schema**: `s`
- **Tables**: `DateRange` (base) and `DateTable` (complete)

## 1. Define Date Range

In [0]:
# Configuration: date range of the table
# Format: dd.mm.yyyy (German format, parsed in the notebook)
start_date = "01.01.1900"
end_date   = "31.12.2222"

## 2. Create Schema
Creates the `s` schema in the Unity Catalog `thekitchen` if it does not already exist.

In [0]:
# Create schema 's' in catalog 'thekitchen' if it does not exist
spark.sql("CREATE SCHEMA IF NOT EXISTS thekitchen.s")
print("Schema 'thekitchen.s' created or already exists.")

## 3. Generate Date Table and Write to Unity Catalog

This cell:
- Parses the date values from the German format (dd.mm.yyyy)
- Builds a list of all days between the start and end date
- Converts these into a Spark DataFrame
- Writes the data as a Delta table to `thekitchen.s.DateRange`

In [0]:
from datetime import datetime, timedelta
from pyspark.sql.functions import col
from pyspark.sql.types import DateType

# Parse the date values from the German format (dd.mm.yyyy) into Python date objects
start_dt = datetime.strptime(start_date, "%d.%m.%Y").date()
end_dt = datetime.strptime(end_date, "%d.%m.%Y").date()

# Generate all dates between start and end
date_list = []
current_date = start_dt
while current_date <= end_dt:
    date_list.append((current_date,))
    current_date += timedelta(days=1)

# Create DataFrame
df_dates = spark.createDataFrame(date_list, ["DateRange"])

# Write to Delta table (overwrite mode)
df_dates.write.format("delta").mode("overwrite").saveAsTable("thekitchen.s.DateRange")

print(f"DateRange table created with {df_dates.count():,} dates from {start_dt} to {end_dt}")

## 4. Create the Extended Date Table with All Attributes

This Spark SQL query:
- Reads the base date table `thekitchen.s.DateRange`
- Calculates all additional columns for analytics:
  - **Calendar hierarchies**: year, month, quarter, calendar week
  - **Offsets**: differences from today (days, months, years)
  - **Workdays**: flags for weekends, workdays, and business days
  - **Sort helper fields**: for correct sorting in BI tools
- Writes the result to the final table `thekitchen.s.DateTable`

The table can then be used in dashboards and BI tools as a date dimension for time intelligence.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE thekitchen.s.DateTable AS
SELECT 
    D.DateRange AS Date,
    
    -- === Calendar hierarchies ===
    DATE_TRUNC('month', D.DateRange) AS CY_Month,
    DATE_TRUNC('year', D.DateRange) AS CY_Year,
    
    -- === Months ===
    MAKE_DATE(1900, MONTH(D.DateRange), 1) AS Months_Long,
    MAKE_DATE(1900, MONTH(D.DateRange), 1) AS Months,
    
    -- === Quarters ===
    CONCAT('Q', QUARTER(D.DateRange), '/', YEAR(D.DateRange)) AS QuarterYear,
    CONCAT('Q', QUARTER(D.DateRange)) AS Quarters,
    DATE_TRUNC('quarter', D.DateRange) AS QuarterYearSort,
    MAKE_DATE(1900, MONTH(DATE_TRUNC('quarter', D.DateRange)), 1) AS QuartersSort,
    
    -- === Offsets (differences from today) ===
    YEAR(D.DateRange) - YEAR(CURRENT_DATE()) AS OffsetYear,
    CAST(MONTHS_BETWEEN(D.DateRange, CURRENT_DATE()) AS INT) AS OffsetMonth,
    DATEDIFF(D.DateRange, CURRENT_DATE()) AS OffsetDay,
    
    -- === Additional date attributes ===
    D.DateRange AS DateLong,
    CASE WHEN D.DateRange = LAST_DAY(D.DateRange) THEN TRUE ELSE FALSE END AS Is_EoM,
    
    -- === Calendar weeks ===
    CONCAT('CW ', LPAD(WEEKOFYEAR(D.DateRange), 2, '0')) AS Calendar_Weeks,
    WEEKOFYEAR(D.DateRange) AS Calendar_Weeks_SORT,
    
    -- === Weekdays ===
    -- In Spark: 1=Sunday, 2=Monday, ..., 7=Saturday
    MAKE_DATE(1900, 1, DAYOFWEEK(D.DateRange)) AS WeekdayDate,
    MAKE_DATE(1900, 1, DAYOFWEEK(D.DateRange)) AS WeekdayDateLong,
    DAYOFWEEK(D.DateRange) AS WeekDayNum,
    
    -- === Additional numeric attributes ===
    DAYOFYEAR(D.DateRange) AS DayOfYear,
    MONTH(D.DateRange) AS MonthNumber,
    
    -- === Workday flags ===
    -- Monday (2) through Friday (6) = workday
    CASE 
        WHEN DAYOFWEEK(D.DateRange) BETWEEN 2 AND 6 THEN 'Workday'
        ELSE 'Weekend'
    END AS Workday,
    
    -- Monday (2) through Saturday (7) = business day  
    CASE 
        WHEN DAYOFWEEK(D.DateRange) BETWEEN 2 AND 7 THEN 'Business day'
        ELSE 'Rest day'
    END AS BusinessDay
    
FROM thekitchen.s.DateRange AS D
""")

print("DateTable created successfully in thekitchen.s.DateTable")